# 03: Runtime benchmark (engineering only)

This notebook compares only **validated** implementations: numpy CPU, numba CPU, and a GPU backend only if the
frozen bundle implements one. A backend is timed only after the bundle self-test passes on it. Its outputs are
then compared with the numpy reference: candidate sets and ranks must match exactly, and features and scores
within 1e-9.

The benchmark runs the fixed self-test fixture first. It can then run an optional deterministic sample of the
test spectra (`N_SAMPLE`). Results **never** change model, aggregator or backend selection.

On Colab, run it after `01_colab_inference.ipynb` in the same session; it reuses the local bundle copy. On Kaggle,
attach the same datasets as `02`.

In [ ]:
# CELL 1 -- configuration + locate the orchestration package and the (local) bundle
import json, os, sys, time
from datetime import datetime, timezone
from pathlib import Path

REQUIRE_GPU = False          # True: stop if no validated GPU backend exists (benchmarking only)
N_SAMPLE = 0                 # 0 = fixture only; e.g. 200 = also the first 200 test spectra (sorted by spectrum_id)
ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    INPUT_ROOT, OUT_DIR = Path('/kaggle/input'), Path('/kaggle/working')
    SRC = sorted({p.parent.parent for p in INPUT_ROOT.rglob('casmi_runtime/__init__.py') if p.parent.parent.name == 'src'})[0]
else:
    INPUT_ROOT, OUT_DIR = Path('/content/casmi_runtime/input'), Path('/content/casmi_runtime/benchmarks')
    SRC = Path('/content/enveda-casmi-runtime/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
BUNDLE_DIR = [p.parent for p in INPUT_ROOT.rglob('manifest.json')
              if str(json.loads(p.read_text(encoding='utf-8')).get('bundle_format', '')).startswith('casmi-modeA-bundle')]
assert len(BUNDLE_DIR) == 1, f'expected one local bundle under {INPUT_ROOT} (Colab: run 01 first), found {BUNDLE_DIR}'
BUNDLE_DIR = BUNDLE_DIR[0]
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('bundle', BUNDLE_DIR, '| orchestration', SRC)

In [ ]:
# CELL 2 -- accelerator + FROZEN code + verified bundle
from casmi_runtime.accelerator import detect_accelerator
from casmi_runtime.benchmark import available_backends, parity_vs_reference, time_backend
from casmi_runtime.frozen import load_frozen_code
ACCEL = detect_accelerator()
FROZEN = load_frozen_code(BUNDLE_DIR)
BUNDLE = FROZEN.pipeline.Bundle(BUNDLE_DIR, verify=True)
print(ACCEL['device'], ACCEL['gpu_name'], '| bundle', BUNDLE.manifest.get('bundle_version'), BUNDLE.config['CONFIG_HASH'], '| code', FROZEN.source)

In [ ]:
# CELL 3 -- validate every candidate backend with the bundle self-test; only PASSing backends are benchmarked
CANDIDATES, GPU_SKIP_REASON = available_backends(FROZEN, ACCEL, BUNDLE.config)
print('GPU backend:', 'candidate' if GPU_SKIP_REASON is None else f'not benchmarked -- {GPU_SKIP_REASON}')
VALIDATED, SELFTEST = [], {}
for be, why in CANDIDATES:
    try:
        res = FROZEN.selftest.run_selftest(BUNDLE, be)
    except Exception as e:
        res = {'passed': False, 'error': f'{type(e).__name__}: {e}'}
    SELFTEST[be] = res
    print(f"{be:8s} self-test {'PASS' if res.get('passed') else 'FAIL'} ({why})")
    if res.get('passed'):
        VALIDATED.append(be)
assert FROZEN.backend.NUMPY in VALIDATED, 'the numpy reference backend failed the self-test -- STOP'
if REQUIRE_GPU and not any(be not in (FROZEN.backend.NUMBA, FROZEN.backend.NUMPY) for be in VALIDATED):
    raise RuntimeError(f'REQUIRE_GPU=True but no validated GPU backend ({GPU_SKIP_REASON})')

In [ ]:
# CELL 4 -- fixed fixture benchmark (the self-test spectra), numpy = reference
import pandas as pd
FEATS = list(BUNDLE.ranker.feature_names)
rows = pd.read_parquet(BUNDLE_DIR / 'selftest' / 'queries.parquet').to_dict('records')
RESULTS = []
ref_metrics, ref_out = time_backend(BUNDLE, FROZEN, FROZEN.backend.NUMPY, rows)
for be in VALIDATED:
    m, out = (ref_metrics, ref_out) if be == FROZEN.backend.NUMPY else time_backend(BUNDLE, FROZEN, be, rows)
    RESULTS.append({'set': 'fixture', **m, **parity_vs_reference(out, ref_out, FEATS)})
display(pd.DataFrame(RESULTS)[['set', 'backend', 'n_spectra', 'runtime_seconds', 'spectra_per_sec', 'candidate_pairs_per_sec',
                                'similarity_evaluations_per_sec', 'rss_gb_after', 'gpu_memory_used_mb', 'parity']])

In [ ]:
# CELL 5 -- optional deterministic sample of the test spectra through the production per-spectrum path
if N_SAMPLE:
    import pyarrow.parquet as pq
    tests = [p for p in INPUT_ROOT.rglob('*.parquet') if BUNDLE_DIR not in p.parents
             and set(FROZEN.pipeline.TEST_REQUIRED) <= set(pq.read_schema(p).names)]
    assert len(tests) == 1, tests
    sample_rows = pd.read_parquet(tests[0]).sort_values('spectrum_id').head(N_SAMPLE).to_dict('records')
    infer = lambda b, row: FROZEN.pipeline.process_spectrum(b, row, FROZEN.pipeline.RunLog())[0]
    s_ref_m, s_ref = time_backend(BUNDLE, FROZEN, FROZEN.backend.NUMPY, sample_rows, infer_fn=infer)
    for be in VALIDATED:
        m, out = (s_ref_m, s_ref) if be == FROZEN.backend.NUMPY else time_backend(BUNDLE, FROZEN, be, sample_rows, infer_fn=infer)
        RESULTS.append({'set': f'test_sample_{N_SAMPLE}', **m, **parity_vs_reference(out, s_ref, FEATS)})
    display(pd.DataFrame(RESULTS)[['set', 'backend', 'n_spectra', 'runtime_seconds', 'spectra_per_sec', 'candidate_pairs_per_sec', 'parity']])

In [ ]:
# CELL 6 -- persist (engineering record only; never read by any selection step)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
rec = {'stamp': stamp, 'platform': 'kaggle' if ON_KAGGLE else 'colab', 'accelerator': ACCEL, 'bundle_version': BUNDLE.manifest.get('bundle_version'),
       'CONFIG_HASH': BUNDLE.config['CONFIG_HASH'], 'gpu_backend_skip_reason': GPU_SKIP_REASON, 'validated_backends': VALIDATED,
       'self_test': SELFTEST, 'results': RESULTS, 'note': 'engineering benchmark; does not change model / aggregator / backend policy'}
path = OUT_DIR / f'benchmark_{stamp}.json'
path.write_text(json.dumps(rec, indent=2, default=str), encoding='utf-8')
print('saved', path)